In [1]:
using SpeedyWeather, CairoMakie, GLMakie

In [2]:
spectral_grid = SpectralGrid()

SpectralGrid{Spectrum{...}, OctahedralGaussianGrid{...}}
├ Number format: Float32
├ Spectral:      T31 LowerTriangularMatrix
├ Grid:          48-ring OctahedralGaussianGrid, 3168 grid points
├ Resolution:    3.61°, 401km (at 6371km radius)
├ Vertical:      8-layer atmosphere, 2-layer land
└ Architecture:  CPU using Array

In [3]:
model = PrimitiveWetModel(spectral_grid)
simulation = initialize!(model)

Simulation{PrimitiveWetModel}
├ prognostic_variables::PrognosticVariables{...}
├ diagnostic_variables::DiagnosticVariables{...}
└ model::PrimitiveWetModel{...}

## Output variables

In [4]:
model.output

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}}, RingGrids.AnvilLocator{Float32, Vector{Float32}, Vector{Int64}}}
├ path: output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ v: meridional wind [m/s]
 ├ humid: specific humidity [kg/kg]
 ├ temp: temperature [degC]
 ├ u: zonal wind [m/s]
 ├ mslp: mean sea-level pressure [hPa]
 └ vor: relative vorticity [s^-1]

In [5]:
add!(model, SpeedyWeather.RadiationOutput()...)

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}}, RingGrids.AnvilLocator{Float32, Vector{Float32}, Vector{Int64}}}
├ path: output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ sru: Surface shortwave radiation up [W/m^2]
 ├ temp: temperature [degC]
 ├ srd: Surface shortwave radiation down [W/m^2]
 ├ mslp: mean sea-level pressure [hPa]
 ├ vor: relative vorticity [s^-1]
 ├ osr: Outgoing shortwave radiation [W/m^2]
 ├ v: meridional wind [m/s]
 ├ u: zonal wind [m/s]
 ├ albedo: albedo [1]
 ├ lrd: Surface longwave radiation down [W/m^2]
 ├ humid: specific humidity [kg/kg]
 ├ olr: Outgoing longwave radiation [W/m^2]
 └ lru: Surfa

In [6]:
add!(model, SpeedyWeather.SurfaceFluxesOutput()...)

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}}, RingGrids.AnvilLocator{Float32, Vector{Float32}, Vector{Int64}}}
├ path: output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ slf: Surface latent heat flux (positive up) [W/m^2]
 ├ sru: Surface shortwave radiation up [W/m^2]
 ├ temp: temperature [degC]
 ├ srd: Surface shortwave radiation down [W/m^2]
 ├ mslp: mean sea-level pressure [hPa]
 ├ vor: relative vorticity [s^-1]
 ├ osr: Outgoing shortwave radiation [W/m^2]
 ├ v: meridional wind [m/s]
 ├ u: zonal wind [m/s]
 ├ albedo: albedo [1]
 ├ lrd: Surface longwave radiation down [W/m^2]
 ├ shf: Surface sensible heat flux (po

In [7]:
simulation.diagnostic_variables.physics.sensible_heat_flux

3168-element, 48-ring OctahedralGaussianField{Float32, 1} as Array on CPU
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 ⋮
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0

In [8]:
simulation.diagnostic_variables.physics.surface_latent_heat_flux

3168-element, 48-ring OctahedralGaussianField{Float32, 1} as Array on CPU
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 ⋮
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0

In [9]:
# run!(simulation, period=Day(30))
run!(simulation, period=Day(365))

In [10]:
heatmap(simulation.diagnostic_variables.physics.sensible_heat_flux)

In [11]:
simulation.prognostic_variables.clock

Clock
├ time::DateTime = 2000-12-31T00:00:00
├ start::DateTime = 2000-01-01T00:00:00
├ period::Second = 31536000 seconds
├ timestep_counter::Int64 = 13140
├ n_timesteps::Int64 = 13140
└ Δt::Millisecond = 2400000 milliseconds

In [12]:
function calc_global_sum(field, model)
    a00 = real(transform(field)[1])
    mean_per_m2 = a00 / model.spectral_transform.norm_sphere
    total_W = mean_per_m2 * (4*pi*model.planet.radius^2)
    return total_W
end


calc_global_sum (generic function with 1 method)

In [13]:
function calc_global_mean(field, model)
    a00 = real(transform(field)[1])
    return a00 / model.spectral_transform.norm_sphere    
end
# mean_per_m2 = a00 / model.spectral_transform.norm_sphere

calc_global_mean (generic function with 1 method)

In [14]:
mean_SHF = calc_global_mean(simulation.diagnostic_variables.physics.sensible_heat_flux, model)

27.636118f0

In [15]:
mean_LHF = calc_global_mean(simulation.diagnostic_variables.physics.surface_latent_heat_flux, model)

43.161327f0

In [16]:
simulation.diagnostic_variables.physics

PhysicsVariables
├ grid: OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}
├ ocean: DynamicsVariablesOcean{Float32, Array, OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Field{Float32, 1, Vector{Float32}, OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ land: DynamicsVariablesLand{Float32, Array, OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Field{Float32, 1, Vector{Float32}, OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ rain_large_scale: 3168-element, 48-ring Field{Float32}
├ rain_convection: 3168-element, 48-ring Field{Float32}
├ snow_large_scale: 3168-element, 48-ring Field{Float32}
├ snow_convection: 3168-element, 48-ring Field{Float32}
├ total_precipitation_rate: 3168-element, 48-ring Field{Float32}
├ cloud_top: 3168-element, 48-ring Field{Float32}

In [17]:
function calc_trenberth_variables(simulation, model; SumFlag::Bool=false)
    # this is a function to calculate the variables we need to plot the Trenberth diagram.
    # simulation -> the SpeedyWeather simulation output.
    # model -> the SpeedyWeather model structure.
    # SumFlag -> false for clculating using the area mean of the fluxes [W/m^2] and true for calculating for the global sum [W]. 

        # put all fields in a Dict
    fields = Dict(
        :LHF   => simulation.diagnostic_variables.physics.surface_latent_heat_flux,
        :SHF   => simulation.diagnostic_variables.physics.sensible_heat_flux,
        :SSRU  => simulation.diagnostic_variables.physics.surface_shortwave_up,
        :SLRU  => simulation.diagnostic_variables.physics.surface_longwave_up,
        :SSRD  => simulation.diagnostic_variables.physics.surface_shortwave_down,
        :SLRD  => simulation.diagnostic_variables.physics.surface_longwave_down,
        :OSR   => simulation.diagnostic_variables.physics.outgoing_shortwave_radiation,
        :OLR   => simulation.diagnostic_variables.physics.outgoing_longwave_radiation,
        :albedo => simulation.diagnostic_variables.physics.albedo
    )

    # initialize result container
    results = Dict{Symbol, Float64}()

    # pick the function once
    calcfun = SumFlag ? calc_global_sum : calc_global_mean

    for (name, field) in fields
        try
            results[name] = calcfun(field, model)
        catch err
            # more informative error handling
            @warn "Could not compute $name: $err"
            results[name] = NaN
        end
    end

    # optional derived Trenberth terms (example: ASR, surface net radiation)
    # note: sign conventions may vary in your model; adapt as needed
    # ASR (absorbed shortwave at TOA) = incoming_TOA - reflected_TOA
    # If you only have OSR as outgoing shortwave at TOA, and you know S_in_TOA:
    # results[:ASR] = S_in_global_total - results[:OSR]  # only if you have S_in_TOA
    
    # Simple surface net (down - up) :
    results[:SW_net_sfc] = results[:SSRD] - results[:SSRU]    # W/m2 or W
    results[:LW_net_sfc] = results[:SLRD] - results[:SLRU]
    results[:surface_net]  = results[:SW_net_sfc] + results[:LW_net_sfc] - results[:LHF] - results[:SHF]

    return results

    return results
end


calc_trenberth_variables (generic function with 1 method)

In [18]:
R_mean = calc_trenberth_variables(simulation, model; SumFlag=false)  # global means

Dict{Symbol, Float64} with 12 entries:
  :LW_net_sfc  => -237.881
  :OLR         => 301.847
  :SSRD        => 341.257
  :SLRD        => 0.0
  :SHF         => 27.6361
  :SSRU        => 27.2284
  :SLRU        => 237.881
  :LHF         => 43.1613
  :albedo      => 0.116091
  :SW_net_sfc  => 314.029
  :OSR         => 27.2284
  :surface_net => 5.35018

#### adding callbacks: 

In [19]:
# --------------------------
# helper: compute Trenberth diagnostics from diagn + model
# --------------------------
function calc_trenberth_from_diagn(diagn, model; SumFlag::Bool=false)
    fields = Dict(
        :LHF   => diagn.physics.surface_latent_heat_flux,
        :SHF   => diagn.physics.sensible_heat_flux,
        :SSRU  => diagn.physics.surface_shortwave_up,
        :SLRU  => diagn.physics.surface_longwave_up,
        :SSRD  => diagn.physics.surface_shortwave_down,
        :SLRD  => diagn.physics.surface_longwave_down,
        :OSR   => diagn.physics.outgoing_shortwave_radiation,
        :OLR   => diagn.physics.outgoing_longwave_radiation,
        :albedo=> diagn.physics.albedo
    )

    # spectral helpers (ℓ=0 → global mean; multiply by area for total)
    function calc_global_mean(field)
        a = transform(field)              # model transform -> spectral coeffs
        a00 = real(a[1])                  # index 1 == ℓ=0,m=0 (SpeedyWeather layout)
        return a00 / model.spectral_transform.norm_sphere
    end
    function calc_global_sum(field)
        mean_val = calc_global_mean(field)
        area = 4π * model.planet.radius^2
        return mean_val * area
    end

    calcfun = SumFlag ? calc_global_sum : calc_global_mean

    results = Dict{Symbol, Float64}()
    for (k, f) in fields
        try
            results[k] = Float64(calcfun(f))
        catch err
            @warn "calc_trenberth_from_diagn: could not compute $k: $err"
            results[k] = NaN
        end
    end

    # derived surface/Trenberth terms (adjust sign convention as needed)
    results[:SW_net_sfc]  = results[:SSRD] - results[:SSRU]
    results[:LW_net_sfc]  = results[:SLRD] - results[:SLRU]
    results[:surface_net] = results[:SW_net_sfc] + results[:LW_net_sfc] - results[:LHF] - results[:SHF]

    return results
end


calc_trenberth_from_diagn (generic function with 1 method)

In [20]:
# --- callback type ---
Base.@kwdef mutable struct TrenberthCallback <: SpeedyWeather.AbstractCallback
    timestep_counter::Int = 0
    data::Dict{Symbol, Vector{Float64}} = Dict{Symbol, Vector{Float64}}()
    times::Vector{Float64} = Float64[]
    SumFlag::Bool = false
end


TrenberthCallback

In [21]:
# convenience generator: create a recorder pre-filled with keys and preallocated arrays
function TrenberthCallback(;vars = [:LHF,:SHF,:SSRU,:SLRU,:SSRD,:SLRD,:OSR,:OLR,:albedo,:SW_net_sfc,:LW_net_sfc,:surface_net],
                             SumFlag::Bool=false,
                             nsteps::Int=0)
    d = Dict{Symbol, Vector{Float64}}()
    # pre-allocate vectors length nsteps+1 if nsteps>0, otherwise empty and will push
    for v in vars
        d[v] = nsteps > 0 ? Vector{Float64}(undef, nsteps + 1) : Float64[]
    end
    times = nsteps > 0 ? Vector{Float64}(undef, nsteps + 1) : Float64[]
    return TrenberthCallback(0, d, times, SumFlag)
end

TrenberthCallback

In [22]:
# --- initialize! called once before run (populate sizes and initial state) ---
function SpeedyWeather.initialize!(cb::TrenberthCallback,
                                   progn::PrognosticVariables,
                                   diagn::DiagnosticVariables,
                                   model::AbstractModel)
    # Try to get nsteps, but if it doesn't work, just start with empty vectors
    try
        nsteps = progn.clock.nsteps
        # if our data dict vectors are empty or wrong size, (re)allocate
        for (k, v) in cb.data
            if isempty(v) || length(v) != nsteps + 1
                cb.data[k] = Vector{Float64}(undef, nsteps + 1)
            end
        end
        if isempty(cb.times) || length(cb.times) != nsteps + 1
            cb.times = Vector{Float64}(undef, nsteps + 1)
        end
    catch
        # If we can't get nsteps, just use dynamic push mode
        @info "Could not determine nsteps, using dynamic push mode"
    end

    # set counter to 1 and store initial conditions
    cb.timestep_counter = 1
    t0 = progn.clock.time
    # compute initial values using diagn
    res0 = calc_trenberth_from_diagn(diagn, model; SumFlag=cb.SumFlag)
    for (k, v) in res0
        if haskey(cb.data, k)
            if length(cb.data[k]) > 0
                cb.data[k][1] = v
            else
                push!(cb.data[k], v)
            end
        else
            cb.data[k] = [v]
        end
    end
    if length(cb.times) > 0
        cb.times[1] = float(t0)
    else
        push!(cb.times, float(t0))
    end
    return nothing
end

In [23]:
# --- callback! called every step (after the step completes) ---
function SpeedyWeather.callback!(cb::TrenberthCallback,
                                 progn::PrognosticVariables,
                                 diagn::DiagnosticVariables,
                                 model::AbstractModel)
    # increment step index
    cb.timestep_counter += 1
    i = cb.timestep_counter
    # compute current diagnostics
    res = calc_trenberth_from_diagn(diagn, model; SumFlag=cb.SumFlag)
    # store them into preallocated arrays (assumes cb.data already has keys)
    for (k, v) in res
        if !haskey(cb.data, k)
            # if a new key appears, create and fill (resize existing to fit previous slots)
            oldlen = i - 1
            newvec = Vector{Float64}(undef, length(cb.times))
            fill!(newvec, NaN)
            newvec[i] = v
            cb.data[k] = newvec
        else
            cb.data[k][i] = v
        end
    end
    # record model time
    cb.times[i] = float(progn.clock.current_time)
    return nothing
end

In [24]:
# --- finalize (optional) ---
SpeedyWeather.finalize!(cb::TrenberthCallback, args...) = nothing

adding the calbak to the model


In [25]:
# preallocated (fast) if you know nsteps:
# cb = TrenberthCallback(SumFlag=false, grow=false, nsteps=1000)

# OR dynamic push mode (flexible):
cb = TrenberthCallback(SumFlag=true,  grow=true,  nsteps=0)


MethodError: MethodError: no method matching TrenberthCallback(; SumFlag::Bool, grow::Bool, nsteps::Int64)
This method does not support all of the given keyword arguments (and may not support any).

Closest candidates are:
  TrenberthCallback(; vars, SumFlag, nsteps) got unsupported keyword argument "grow"
   @ Main c:\Users\Ofer - Personal\Desktop\SpeedyWeather_Project\speedyweather_trenberth_diagram\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X31sZmlsZQ==.jl:2
  TrenberthCallback(!Matched::Int64, !Matched::Dict{Symbol, Vector{Float64}}, !Matched::Vector{Float64}, !Matched::Bool) got unsupported keyword arguments "SumFlag", "grow", "nsteps"
   @ Main c:\Users\Ofer - Personal\Desktop\SpeedyWeather_Project\speedyweather_trenberth_diagram\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X30sZmlsZQ==.jl:3
  TrenberthCallback(!Matched::Any, !Matched::Any, !Matched::Any, !Matched::Any) got unsupported keyword arguments "SumFlag", "grow", "nsteps"
   @ Main c:\Users\Ofer - Personal\De

In [26]:
# A: add into the callbacks dict directly (works if model.callbacks is a Dict-like)
add!(model.callbacks, :trenberth => cb)

UndefVarError: UndefVarError: `cb` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

In [27]:
keys(model.callbacks)              # should include :trenberth
model.callbacks[:trenberth] === cb # should be true


KeyError: KeyError: key :trenberth not found

In [28]:
sim = initialize!(model)   # this will call SpeedyWeather.initialize! on cb
run!(sim, period=Day(30))      # or your usual run invocation
